# RL Group Project: Config A

## Clinical Treatment Optimisation: Sepsis ICU Management

**Master in Data Science & Advanced Analytics — Reinforcement Learning Course**

This project is structured in two stages of increasing complexity.

- In **Configuration A**, you will work with a tabular Sepsis MDP, where the
  state and action spaces are small enough to apply classical RL methods
  directly.

- In **Configuration B**, you will move to a continuous-observation ICU
  environment that is clinically grounded and significantly more challenging.

Three realistic failure modes are present in Configuration B, each reflecting a
real scenario encountered in clinical AI deployments. The first is episodic
observation noise, where monitoring equipment occasionally malfunctions. The
second is episodic missing observations, representing situations where lab
results are simply unavailable for an entire episode. The third is acute
clinical events, which are sudden and irreversible patient deteriorations that
occur independently of any treatment given.

---

### Group Members

Group Q

```
Student 1: Diogo Carvalho - 20221935
Student 2: Ricardo Pereira - 20250343
Student 3: Yehor Malakhov - 20221691
```


# Config A — Tabular Methods


## Setup & Imports


In [20]:
import os
import warnings
from typing import TYPE_CHECKING

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

from config_a import (
    evaluate_deterministic_policy,
    greedy_policy_from_q,
    policy_iteration,
    train_q_learning,
    tune_q_learning,
)

#  Import constants and env factory from env_setup.py
from envs.env_setup import (
    GAMMA,
    N_ACTIONS,
    N_STATES,
    STATE_DIED,
    STATE_SURVIVED,
    make_sepsis_env,
)

if TYPE_CHECKING:
    import icu_sepsis

In [ ]:
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

SEED = 42
np.random.seed(SEED)


print(f"ICU-Sepsis-v2 | States: {N_STATES} | Actions: {N_ACTIONS}")
print(
    f"Terminal states: {STATE_SURVIVED} (survived, r=+1)  {STATE_DIED} (died, r=0)"
)
print("Setup complete!")

ICU-Sepsis-v2 | States: 716 | Actions: 25
Terminal states: 714 (survived, r=+1)  713 (died, r=0)
Setup complete!


---

## Explore the Environment

`ICU-Sepsis-v2` is a benchmark MDP constructed from real MIMIC-III patient data.
Each episode represents the trajectory of one ICU patient. The agent observes a
discrete integer state (ranging from 0 to 715) and must select one of 25
treatment actions corresponding to combinations of vasopressor and IV fluid dose
levels. The reward signal is sparse: **+1 at survival, 0 at death, and 0 for all
intermediate steps**, with a discount factor γ = 1.


In [22]:
#  Instantiate and inspect the raw environment
env = make_sepsis_env()
obs, info = env.reset(seed=SEED)

print(f"Observation space : {env.observation_space} discrete integer state")
print(f"Action space      : {env.action_space}")
print(f"Initial state     : {obs}")
print()

#  Extract the full MDP model

# pyright reports wrong type for env.unwrapped
raw: icu_sepsis.ICUSepsisEnv = env.unwrapped  # pyright: ignore[reportAssignmentType]
P: np.ndarray[tuple[int, int, int], np.dtype[np.float64]] = (
    raw._tx_mat
)  # shape (716, 25, 716) — P[s,a,s'] = P(s'|s,a)
R_sasp: np.ndarray[tuple[int, int, int], np.dtype[np.float64]] = (
    raw._r_mat
)  # (716, 25, 716) — R[s, a, s']
R: np.ndarray[tuple[int, int], np.dtype[np.float64]] = (P * R_sasp).sum(
    axis=2
)  # (716, 25)      — E[r | s, a]

print(f"Transition matrix P : {P.shape}  (S × A × S')")
print(f"Reward matrix R     : {R.shape}  (S × A)")
print(f"Reward range        : [{R.min():.3f}, {R.max():.3f}]")
print()

Observation space : Discrete(716) discrete integer state
Action space      : Discrete(25)
Initial state     : 559

Transition matrix P : (716, 25, 716)  (S × A × S')
Reward matrix R     : (716, 25)  (S × A)
Reward range        : [-0.020, 0.708]



In [23]:
def run_random_baseline(n_episodes=1000, seed=SEED):
    np.random.seed(seed)
    env_eval = make_sepsis_env()
    returns, lengths = [], []
    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=np.random.randint(100_000))
        total_r, steps, done = 0.0, 0, False
        while not done:
            obs, r, te, tr, _ = env_eval.step(env_eval.action_space.sample())
            total_r += r
            steps += 1
            done = te or tr
        returns.append(total_r)
        lengths.append(steps)
    env_eval.close()
    return np.array(returns), np.array(lengths)


rand_returns, rand_lengths = run_random_baseline()
survival_rate = float(np.mean(rand_returns > 0)) * 100

print(f"Random agent ({len(rand_returns)} episodes):")
print(f"  Mean return    : {np.mean(rand_returns):.4f}")
print(f"  Survival rate  : {survival_rate:.1f}%")
print(f"  Mean ep length : {np.mean(rand_lengths):.1f} steps")
print()
print("All Config A algorithms must beat the random baseline.")

Random agent (1000 episodes):
  Mean return    : 0.5761
  Survival rate  : 68.1%
  Mean ep length : 10.4 steps

All Config A algorithms must beat the random baseline.


---

With 716 discrete states and 25 actions, the Q-table has shape `(716, 25)`,
totalling 17,900 entries. This size is entirely manageable in memory, which is
precisely what motivates the use of tabular algorithms here.


### Policy Iteration


In [30]:
pi_policy, pi_values = policy_iteration(
    P,
    R,
    gamma=GAMMA,
    max_policy_iter=1_000,
)
pi_expected_return = float(np.dot(raw._d_0, pi_values))
pi_returns, pi_lengths = evaluate_deterministic_policy(
    make_sepsis_env(),
    pi_policy,
    n_episodes=50_000,
    seed=SEED,
)
pl.DataFrame(pi_returns).write_csv("logs/pi_mean_return_eval.csv")
pl.DataFrame(pi_lengths).write_csv("logs/pi_episode_length_eval.csv")

Policy Iteration:   0%|          | 3/1000 [00:00<01:57,  8.49it/s]


Converged after 4 iterations.


In [26]:
print("Policy Iteration")
print(f"  Mean return: {pi_returns.mean():.4f}")
print(
    f"  STD of returns of survivors: {np.array([p for p in pi_returns if p > 0]).std():.4f}"
)
print(
    f"  STD of returns of non-survivors: {np.array([p for p in pi_returns if p <= 0]).std():.4f}"
)
print(f"  Survival rate: {(pi_returns > 0).mean() * 100:.1f}%")
print(f"  Mean episode length: {pi_lengths.mean():.1f} steps")

Policy Iteration
  Mean return: 0.7868
  STD of returns of survivors: 0.0474
  STD of returns of non-survivors: 0.0434
  Survival rate: 82.5%
  Mean episode length: 9.8 steps


### Q-Learning


In [ ]:
q_study = tune_q_learning(
    make_sepsis_env,
    n_states=N_STATES,
    n_actions=N_ACTIONS,
    gamma=GAMMA,
    n_trials=50,
    train_episodes=400_000,
    validation_episodes=15000,
    seed=SEED,
)

[I 2026-06-03 21:36:55,617] A new study created in memory with name: no-name-4cd8b180-1295-46d9-8f8b-4ad9eacc6777
  2%|▏         | 1/50 [01:09<56:52, 69.65s/it]

[I 2026-06-03 21:38:05,265] Trial 0 finished with values: [0.6738340016980655, 0.734] and parameters: {'alpha': 0.0005753773794117186, 'epsilon_end': 0.07114476009343425, 'exploration_fraction': 0.7319939418114051}.


  4%|▍         | 2/50 [02:18<55:31, 69.40s/it]

[I 2026-06-03 21:39:14,495] Trial 1 finished with values: [0.6661243350130817, 0.7288] and parameters: {'alpha': 0.006502468545951014, 'epsilon_end': 0.00029380279387035364, 'exploration_fraction': 0.15599452033620265}.


  6%|▌         | 3/50 [03:27<54:06, 69.07s/it]

...

[I 2026-06-03 22:34:51,641] Trial 47 finished with values: [0.6787366679514758, 0.7937333333333333] and parameters: {'alpha': 0.4623535130871452, 'epsilon_end': 0.00033854332664918303, 'exploration_fraction': 0.805463123985254}.


 98%|█████████▊| 49/50 [59:05<01:12, 72.58s/it]

[I 2026-06-03 22:36:00,737] Trial 48 finished with values: [0.6664828350916194, 0.7283333333333334] and parameters: {'alpha': 0.0015382104663928986, 'epsilon_end': 0.0003886647132025148, 'exploration_fraction': 0.8844009995795085}.


100%|██████████| 50/50 [1:00:13<00:00, 72.27s/it]

[I 2026-06-03 22:37:09,255] Trial 49 finished with values: [0.6751493351178555, 0.7318] and parameters: {'alpha': 4.398078645271038e-05, 'epsilon_end': 0.050518204688338515, 'exploration_fraction': 0.772774605551797}.


In [ ]:
q_study.best_trials

[FrozenTrial(number=33, state=<TrialState.COMPLETE: 1>, values=[0.7073938347674595, 0.8098], datetime_start=datetime.datetime(2026, 6, 3, 22, 16, 39, 57349), datetime_complete=datetime.datetime(2026, 6, 3, 22, 18, 0, 65427), params={'alpha': 0.18484003928169826, 'epsilon_end': 0.00012850334707145372, 'exploration_fraction': 0.6712650723031417}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'alpha': FloatDistribution(high=0.5, log=True, low=1e-05, step=None), 'epsilon_end': FloatDistribution(high=0.1, log=True, low=0.0001, step=None), 'exploration_fraction': FloatDistribution(high=1.0, log=False, low=0.0, step=None)}, trial_id=33, value=None)]

In [ ]:
q_study.best_trials[0].params

{'alpha': 0.18484003928169826,
 'epsilon_end': 0.00012850334707145372,
 'exploration_fraction': 0.6712650723031417}

In [ ]:
pl.DataFrame(q_study.best_trials[0].params).write_csv(
    "logs/ql_best_trial_params.csv"
)

In [27]:
qlearning_hyperparameters = pl.read_csv("logs/ql_best_trial_params.csv")

In [32]:
q_env = make_sepsis_env()
q_table, q_returns, q_lengths = train_q_learning(
    q_env,
    n_states=N_STATES,
    n_actions=N_ACTIONS,
    gamma=GAMMA,
    episodes=400_000,
    alpha=qlearning_hyperparameters["alpha"][0],
    epsilon_start=1.0,
    epsilon_end=qlearning_hyperparameters["epsilon_end"][0],
    exploration_fraction=qlearning_hyperparameters["exploration_fraction"][0],
    seed=SEED,
)
q_env.close()


q_policy = greedy_policy_from_q(q_table)
q_returns_eval, q_lengths_eval = evaluate_deterministic_policy(
    make_sepsis_env(), q_policy, n_episodes=15000, seed=SEED
)
pl.DataFrame(q_returns).write_csv("logs/ql_episode_returns_train.csv")
pl.DataFrame(q_returns_eval).write_csv("logs/ql_mean_return_eval.csv")
pl.DataFrame(q_lengths).write_csv("logs/ql_episode_length_train.csv")
pl.DataFrame(q_lengths_eval).write_csv("logs/ql_episode_length_eval.csv")

In [29]:
q_ma = pl.Series(q_returns).rolling_mean(window_size=5000)
print("Q-learning")
print(f"  Mean return: {q_returns_eval.mean():.4f}")
print(
    f"  STD of returns of survivors: {np.array([p for p in q_returns_eval if p > 0]).std():.4f}"
)
print(
    f"  STD of returns of non-survivors: {np.array([p for p in q_returns_eval if p <= 0]).std():.4f}"
)
print(f"  Survival rate: {(q_returns_eval > 0).mean() * 100:.1f}%")
print(f"  Mean episode length: {q_lengths_eval.mean():.1f} steps")

Q-learning
  Mean return: 0.7074
  STD of returns of survivors: 0.1083
  STD of returns of non-survivors: 0.0930
  Survival rate: 81.0%
  Mean episode length: 12.2 steps
